# Compare GPU Roofline Results

This notebook reloads and re-parses Nsight Compute report files for multiple GPUs, normalizes the data, and compares per-kernel roofline metrics (traffic, FLOPs, arithmetic intensity, and timing) across devices for both CUDA and OpenMP builds.

In [12]:
import pandas as pd
import os
import glob
import numpy as np
import seaborn as sns
import json
import matplotlib.pyplot as plt
import re
import sys
from tqdm import tqdm

sys.path.append('../')
from utils import *
from gatherData import _parse_ncu_report, roofline_results_to_df, calc_roofline_data 

from tqdm.contrib.concurrent import process_map
from functools import partial
import os
from os import path


from sass_helper import *

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [13]:
# kernel key and metrics
key_cols = ['source', 'Kernel Name', 'exeArgs', 'Block Size', 'Grid Size', 'model_type']
device_col = 'device'

groupings = key_cols + [device_col]
metrics = ['SP_FLOP', 'DP_FLOP', 'HP_FLOP', 'INTOP', 'traffic',
           'bytesRead', 'bytesWrite', 'bytesTotal',
           'dpAI', 'spAI', 'hpAI',
           'dpPerf', 'spPerf', 'hpPerf', 'xtime']

log_metrics = ['traffic', 'xtime']

# all cols
all_cols = groupings + metrics + ['sample']

# markers we should ignore / drop kernels containing these from the dataset
#library_markers = [ 'cub::', 'thrust::', '__cuda_' ]

In [14]:
df = pd.read_csv('all-NCU-GPU-Data.csv', low_memory=False)
print(df.shape)
print(df.columns)


# let's rename the device column
def rename_devices(x):
    if '3080' in x:
        return '3080'
    elif 'A100' in x:
        return 'A100'
    elif 'A10' in x:
        return 'A10'
    elif 'H100' in x:
        return 'H100'
    else:
        raise ValueError(f'Unknown device name in {x}')

df['device'] = df['device'].apply(lambda x: rename_devices(x))


(20044, 2659)
Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'smsp__sass_inst_executed_op_shared.sum.pct_of_peak_sustained_elapsed',
       'smsp__sass_inst_executed_op_shared_gmma.sum',
       'smsp__sass_inst_executed_op_shared_gmma.sum.pct_of_peak_sustained_elapsed',
       'smsp__sass_inst_executed_op_tma_ld.sum',
       'smsp__sass_inst_executed_op_tma_red.sum',
       'smsp__sass_inst_executed_op_tma_st.sum', 'launch_cluster_dim_x',
       'launch_cluster_dim_z', 'launch_cluster_max_active',
       'launch_cluster_dim_y'],
      dtype='object', length=2659)


In [15]:
display(df[all_cols].head(10))

,source,Kernel Name,exeArgs,Block Size,Grid Size,model_type,device,SP_FLOP,DP_FLOP,HP_FLOP,INTOP,traffic,bytesRead,bytesWrite,bytesTotal,dpAI,spAI,hpAI,dpPerf,spPerf,hpPerf,xtime,sample
0,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,3080,0,0,0,5.629983e+08,6.449159e+11,327800576,1797888,329598464,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,511072.0,1
1,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,3080,0,0,0,5.629983e+08,6.425681e+11,327824384,1766784,329591168,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,512928.0,2
2,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,3080,0,0,0,5.629983e+08,6.443553e+11,327800320,1800320,329600640,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,511520.0,3
3,accuracy-omp,__omp_offloading_10305_2b800b0_main_l57,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3080,0,0,0,1.003935e+09,5.279006e+11,327974528,1756416,329730944,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,624608.0,1
4,accuracy-omp,__omp_offloading_10305_2b800b0_main_l57,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3080,0,0,0,1.003935e+09,5.271992e+11,327968896,1762560,329731456,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,625440.0,2
5,accuracy-omp,__omp_offloading_10305_2b800b0_main_l57,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3080,0,0,0,1.003935e+09,5.287411e+11,327975808,1772544,329748352,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,623648.0,3
6,ace-cuda,_Z14calculateForcePA400_A400_dS1_S1_S1_dddddd,100,"(8, 8, 4)","(50, 50, 100)",cuda,3080,378175731,7817840356,0,7.015186e+09,6.507823e+10,764802432,1538377728,2303180160,3.394368,0.164197,0.0,2.208995e+11,1.068566e+10,0.0,35390944.0,1
7,ace-cuda,_Z9allenCahnPA400_A400_dS1_S1_S1_S1_S1_dddddddd,100,"(8, 8, 4)","(50, 50, 100)",cuda,3080,882719027,14814682409,0,1.476101e+10,6.287344e+10,3050602496,508060544,3558663040,4.162991,0.248048,0.0,2.617416e+11,1.559563e+10,0.0,56600416.0,1
8,ace-cuda,_Z21boundaryConditionsPhiPA400_A400_d,100,"(8, 8, 4)","(50, 50, 100)",cuda,3080,0,0,0,1.084807e+09,5.832238e+10,19200,15426432,15445632,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,264832.0,1
9,ace-cuda,_Z15thermalEquationPA400_A400_dS1_S1_S1_ddddd,100,"(8, 8, 4)","(50, 50, 100)",cuda,3080,378358346,7173243211,0,6.858070e+09,8.250915e+10,1779667968,507345664,2287013632,3.136511,0.165438,0.0,2.587908e+11,1.365013e+10,0.0,27718304.0,1


In [16]:
sass = pd.read_csv('imix_data.csv', low_memory=False)
print(sass.shape)
print(sass.columns)
print(sass.columns[68], sass.columns[74])

(14757, 151)
Index(['Kernel Name', 'Program', 'Model', 'SM', 'Num Labels', 'Num References',
       'Num Self References', 'Num Circular Dependencies', 'Num Predicated',
       'Num Lines',
       ...
       'TEX', 'BRXU', 'I2FP', 'F2IP', 'VIADD', 'VIADDMNMX', 'ENDCOLLECTIVE',
       'VIMNMX', 'CGAERRBAR', 'VIMNMX3'],
      dtype='object', length=151)
SEL UIADD3


In [17]:
display(sass.head(10))

,Kernel Name,Program,Model,SM,Num Labels,Num References,Num Self References,Num Circular Dependencies,Num Predicated,Num Lines,Num Constant Math,Num FP16,Num FP32,Num FP64,Num Global Loads,Num Global Stores,Labels,References,OpType_movement,OpType_integer,OpType_synchronization,OpType_control,OpType_floating point,MOV,LOP3,...,UP2UR,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3
0,__cuda_sm20_sqrt_rn_f32_slowpath,vmc,cuda,sm_80,4,0,0,0,15,32,5,0,12,0,0,0,".L_x_10,.L_x_9,.L_x_11,.L_x_59",NaN,4.0,1.0,2.0,13,12.0,4.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,_Z9propagateiiPfS_S_S_S_S_S_S_Pj,vmc,cuda,sm_80,33,6,0,0,9,264,112,4,86,0,12,12,".L_x_36,.L_x_0,.L_x_15,.L_x_16,.L_x_14,.L_x_1,...","__cuda_sm20_sqrt_rn_f32_slowpath,__cuda_sm20_s...",25.0,59.0,16.0,40,90.0,49.0,13.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,_Z10initializePfS_S_S_S_S_S_Pj,vmc,cuda,sm_80,14,3,0,0,3,136,70,4,50,0,1,8,".L_x_6,.L_x_39,.L_x_40,.L_x_38,.L_x_7,.L_x_42,...","__cuda_sm20_sqrt_rn_f32_slowpath,__cuda_sm20_s...",11.0,28.0,6.0,19,54.0,23.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,_Z10zero_statsiPf,vmc,cuda,sm_80,2,0,0,0,0,32,8,1,0,0,0,4,".L_x_48,.L_x_62",NaN,2.0,7.0,0.0,15,1.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,_Z7initranjPj,vmc,cuda,sm_80,2,0,0,0,0,24,6,1,0,0,0,1,".L_x_49,.L_x_63",NaN,2.0,5.0,0.0,12,1.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,_Z15SumWithinBlocksiPKfPf,vmc,cuda,sm_80,8,0,0,0,42,152,65,4,14,0,5,1,".L_x_54,.L_x_53,.L_x_52,.L_x_55,.L_x_51,.L_x_5...",NaN,3.0,64.0,13.0,17,18.0,3.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,_Z15LCG_random_initPj,vmc,cuda,sm_80,2,0,0,0,0,16,3,1,0,0,0,0,".L_x_57,.L_x_65",NaN,0.0,2.0,0.0,10,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,_Z10LCG_randomPj,vmc,cuda,sm_80,2,0,0,0,0,24,4,1,1,0,0,0,".L_x_58,.L_x_66",NaN,2.0,2.0,0.0,14,2.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,__kmpc_get_hardware_thread_id_in_block,sort,omp,sm_80,2,0,0,0,0,16,0,0,0,0,0,0,".L_x_4,.L_x_40",NaN,0.0,0.0,0.0,15,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,__omp_offloading_fd01_183cbd_main_l146,sort,omp,sm_80,16,1,0,0,35,304,118,2,1,0,5,4,".L_x_0,.L_x_17,.L_x_7,.L_x_6,.L_x_12,.L_x_9,.L...",__kmpc_get_hardware_thread_id_in_block,8.0,122.0,27.0,28,3.0,4.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# need to rename some of the columns and make a few new ones to match the NCU data
# some of the rows in the imix data will not appear in the perf counter data, as some functions are __device__ functions that are inlined or called by the __global__ kernels

sass['source'] = sass['Program'] + '-' + sass['Model']

In [19]:
# let's add an sm_arch column to the df

device_to_sm_arch = {
    '3080': 'sm_86',
    'A100': 'sm_80',
    'A10': 'sm_86',
    'H100': 'sm_90'
}   

df['SM'] = df['device'].apply(lambda x: device_to_sm_arch[x])

print(df.shape)

(20044, 2660)


In [20]:

def fix_omp_kernel_name(x):
    if '__omp_offloading' in x:
        # we need to do a regex here to extract the function name after the following 
        # possible string patterns:
        # __omp_offloading_fd01_2c7d38_
        # __omp_offloading_10305_2b800c7_
        match = re.search(r'__omp_offloading_.*?_.*?_(.+)$', x)
        if match:
            return match.group(1)
    return x

df['Kernel Name'] = df['Kernel Name'].apply(lambda x: fix_omp_kernel_name(x))   
sass['Kernel Name'] = sass['Kernel Name'].apply(lambda x: fix_omp_kernel_name(x))

In [21]:
# let's match up the 'source', 'Kernel Name', and 'SM' columns in the perf counter data with the 'source', 'Kernel Name', and 'SM' columns in the imix data, and add the imix metrics to the perf counter dataframe

df_merged = pd.merge(df, sass, on=['source', 'Kernel Name', 'SM'], how='left', suffixes=('', '_sass'))

In [22]:
print(df_merged.shape)
print(df_merged.columns)
print(len(df_merged.columns))

(21121, 2809)
Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'TEX', 'BRXU', 'I2FP', 'F2IP', 'VIADD', 'VIADDMNMX', 'ENDCOLLECTIVE',
       'VIMNMX', 'CGAERRBAR', 'VIMNMX3'],
      dtype='object', length=2809)
2809


In [23]:

display(df_merged.head(15))

,ID,Process ID,Process Name,Host Name,Kernel Name,Context,Stream,Block Size,Grid Size,Device,CC,FBSP.TriageSCG.dramc__read_throughput.avg.pct_of_peak_sustained_elapsed,FBSP.TriageSCG.dramc__throughput.avg.pct_of_peak_sustained_elapsed,FBSP.TriageSCG.dramc__write_throughput.avg.pct_of_peak_sustained_elapsed,FE_B.TriageAC.gr__ctas_launched_queue_sync.sum,LTS.TriageSCG.lts__average_t_sector_hit_rate_realtime.pct,LTS.TriageSCG.lts__throughput.avg.pct_of_peak_sustained_elapsed,SM_A.TriageAC.l1tex__data_pipe_lsu_wavefronts.avg,SM_A.TriageAC.l1tex__lsu_writeback_active.avg,SM_A.TriageAC.sm__cycles_active.avg,SM_A.TriageAC.sm__inst_executed_realtime.avg.per_cycle_active,SM_A.TriageSCG.l1tex__throughput.avg.pct_of_peak_sustained_elapsed,SM_A.TriageSCG.sm__inst_executed_pipe_alu_realtime.avg.pct_of_peak_sustained_elapsed,SM_B.TriageAC.l1tex__t_sector_hit_rate.pct,SM_C.TriageSCG.smsp__inst_executed_pipe_fmaheavy.avg.pct_of_peak_sustained_elapsed,...,UP2UR,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3
0,0.0,566100.0,accuracy,127.0.0.1,_Z15accuracy_kerneliiiPKfPKiPi,1.0,7.0,"(256, 1, 1)","(2048, 1, 1)",0.0,8.6,NaN,NaN,NaN,"2,048",NaN,NaN,"44,338.63","42,483.88","686,209.49",319.61,NaN,12.29,NaN,6.14,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,566220.0,accuracy,127.0.0.1,_Z15accuracy_kerneliiiPKfPKiPi,1.0,7.0,"(256, 1, 1)","(2048, 1, 1)",0.0,8.6,NaN,NaN,NaN,"2,048",NaN,NaN,"44,340.56","42,483.90","687,543.22",318.42,NaN,12.26,NaN,6.12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,566354.0,accuracy,127.0.0.1,_Z15accuracy_kerneliiiPKfPKiPi,1.0,7.0,"(256, 1, 1)","(2048, 1, 1)",0.0,8.6,NaN,NaN,NaN,"2,048",NaN,NaN,"44,344.29","42,483.88","686,880.26",319.07,NaN,12.17,NaN,6.07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,566433.0,accuracy,127.0.0.1,main_l57,1.0,13.0,"(128, 1, 1)","(2048, 1, 1)",0.0,8.6,NaN,NaN,NaN,"2,048",NaN,NaN,"56,740.00","46,188.44","861,213.01",640.42,NaN,17.64,NaN,5.34,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,566547.0,accuracy,127.0.0.1,main_l57,1.0,13.0,"(128, 1, 1)","(2048, 1, 1)",0.0,8.6,NaN,NaN,NaN,"2,048",NaN,NaN,"56,740.71","46,188.41","860,023.71",641.49,NaN,20.88,NaN,6.32,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,566656.0,accuracy,127.0.0.1,main_l57,1.0,13.0,"(128, 1, 1)","(2048, 1, 1)",0.0,8.6,NaN,NaN,NaN,"2,048",NaN,NaN,"56,740.10","46,188.40","859,528.90",641.50,NaN,18.77,NaN,5.68,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,566735.0,ace,127.0.0.1,_Z14calculateForcePA400_A400_dS1_S1_S1_dddddd,1.0,7.0,"(8, 8, 4)","(50, 50, 100)",0.0,8.6,NaN,NaN,NaN,"250,000",NaN,NaN,"1,365,062.16","669,550.59","50,960,093.99",910.45,NaN,3.62,NaN,1.49,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1.0,566735.0,ace,127.0.0.1,_Z9allenCahnPA400_A400_dS1_S1_S1_S1_S1_dddddddd,1.0,7.0,"(8, 8, 4)","(50, 50, 100)",0.0,8.6,NaN,NaN,NaN,"250,000",NaN,NaN,"2,016,748.40","1,403,644.34","81,489,728.04",817.06,NaN,4.64,NaN,1.86,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,2.0,566735.0,ace,127.0.0.1,_Z21boundaryConditionsPhiPA400_A400_d,1.0,7.0,"(8, 8, 4)","(50, 50, 100)",0.0,8.6,NaN,NaN,NaN,"250,000",NaN,NaN,"94,067.76","88,235.29","378,540.38",700.25,NaN,32.81,NaN,21.66,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,3.0,566735.0,ace,127.0.0.1,_Z15thermalEquationPA400_A400_dS1_S1_S1_ddddd,1.0,7.0,"(8, 8, 4)","(50, 50, 100)",0.0,8.6,NaN,NaN,NaN,"25

In [26]:

subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['SM'] == 'sm_90')]

display(subset)


,Kernel Name,Program,Model,SM,Num Labels,Num References,Num Self References,Num Circular Dependencies,Num Predicated,Num Lines,Num Constant Math,Num FP16,Num FP32,Num FP64,Num Global Loads,Num Global Stores,Labels,References,OpType_movement,OpType_integer,OpType_synchronization,OpType_control,OpType_floating point,MOV,LOP3,...,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3,source
10901,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,48,472,407,6,400,0,8,4,".L_x_2,.L_x_8",NaN,5.0,26.0,0.0,17,406.0,5.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda


In [27]:

subset = sass[(sass['source'] == 'geglu-cuda') & 
              (sass['SM'] == 'sm_90')]

display(subset)

,Kernel Name,Program,Model,SM,Num Labels,Num References,Num Self References,Num Circular Dependencies,Num Predicated,Num Lines,Num Constant Math,Num FP16,Num FP32,Num FP64,Num Global Loads,Num Global Stores,Labels,References,OpType_movement,OpType_integer,OpType_synchronization,OpType_control,OpType_floating point,MOV,LOP3,...,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3,source
10899,_Z12geglu_kernelIffLi160ELi5120ELi8ELi2EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,48,480,436,4,400,0,8,4,".L_x_0,.L_x_6",NaN,3.0,40.0,0.0,15,404.0,3.0,19.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10900,_Z12geglu_kernelIffLi160ELi2560ELi8ELi2EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,48,472,432,4,400,0,8,4,".L_x_1,.L_x_7",NaN,3.0,37.0,0.0,10,404.0,3.0,18.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10901,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,48,472,407,6,400,0,8,4,".L_x_2,.L_x_8",NaN,5.0,26.0,0.0,17,406.0,5.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10902,_Z12geglu_kernelIffLi160ELi5120ELi8ELi1EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,24,256,211,5,200,0,4,2,".L_x_3,.L_x_9",NaN,4.0,20.0,0.0,15,205.0,4.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10903,_Z12geglu_kernelIffLi160ELi2560ELi8ELi1EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,24,256,211,5,200,0,4,2,".L_x_4,.L_x_10",NaN,4.0,20.0,0.0,15,205.0,4.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10904,_Z12geglu_kernelIffLi160ELi1280ELi8ELi1EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,24,248,208,5,200,0,4,2,".L_x_5,.L_x_11",NaN,4.0,17.0,0.0,10,205.0,4.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda


In [29]:
print(df_merged.shape)

print(df_merged.head(10))


(21121, 2809)
    ID  Process ID Process Name  Host Name  \
0  0.0    566100.0     accuracy  127.0.0.1   
1  0.0    566220.0     accuracy  127.0.0.1   
2  0.0    566354.0     accuracy  127.0.0.1   
3  0.0    566433.0     accuracy  127.0.0.1   
4  0.0    566547.0     accuracy  127.0.0.1   
5  0.0    566656.0     accuracy  127.0.0.1   
6  0.0    566735.0          ace  127.0.0.1   
7  1.0    566735.0          ace  127.0.0.1   
8  2.0    566735.0          ace  127.0.0.1   
9  3.0    566735.0          ace  127.0.0.1   

                                       Kernel Name  Context  Stream  \
0                   _Z15accuracy_kerneliiiPKfPKiPi      1.0     7.0   
1                   _Z15accuracy_kerneliiiPKfPKiPi      1.0     7.0   
2                   _Z15accuracy_kerneliiiPKfPKiPi      1.0     7.0   
3                                         main_l57      1.0    13.0   
4                                         main_l57      1.0    13.0   
5                                         main_l57   

In [ ]:
# let's replace the NaN values with -1, since these represent performance counter values that were not gathered.

df_merged.fillna(-1, inplace=True)

df_merged.to_csv('all-NCU-GPU-Data-with-IMIX.csv', index=False)

,ID,Process ID,Process Name,Host Name,Kernel Name,Context,Stream,Block Size,Grid Size,Device,CC,FBSP.TriageSCG.dramc__read_throughput.avg.pct_of_peak_sustained_elapsed,FBSP.TriageSCG.dramc__throughput.avg.pct_of_peak_sustained_elapsed,FBSP.TriageSCG.dramc__write_throughput.avg.pct_of_peak_sustained_elapsed,FE_B.TriageAC.gr__ctas_launched_queue_sync.sum,LTS.TriageSCG.lts__average_t_sector_hit_rate_realtime.pct,LTS.TriageSCG.lts__throughput.avg.pct_of_peak_sustained_elapsed,SM_A.TriageAC.l1tex__data_pipe_lsu_wavefronts.avg,SM_A.TriageAC.l1tex__lsu_writeback_active.avg,SM_A.TriageAC.sm__cycles_active.avg,SM_A.TriageAC.sm__inst_executed_realtime.avg.per_cycle_active,SM_A.TriageSCG.l1tex__throughput.avg.pct_of_peak_sustained_elapsed,SM_A.TriageSCG.sm__inst_executed_pipe_alu_realtime.avg.pct_of_peak_sustained_elapsed,SM_B.TriageAC.l1tex__t_sector_hit_rate.pct,SM_C.TriageSCG.smsp__inst_executed_pipe_fmaheavy.avg.pct_of_peak_sustained_elapsed,...,UP2UR,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3
0,0.0,566100.0,accuracy,127.0.0.1,_Z15accuracy_kerneliiiPKfPKiPi,1.0,7.0,"(256, 1, 1)","(2048, 1, 1)",0.0,8.6,-1.0,-1.0,-1.0,"2,048",-1.0,-1.0,"44,338.63","42,483.88","686,209.49",319.61,-1.0,12.29,-1.0,6.14,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,566220.0,accuracy,127.0.0.1,_Z15accuracy_kerneliiiPKfPKiPi,1.0,7.0,"(256, 1, 1)","(2048, 1, 1)",0.0,8.6,-1.0,-1.0,-1.0,"2,048",-1.0,-1.0,"44,340.56","42,483.90","687,543.22",318.42,-1.0,12.26,-1.0,6.12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,566354.0,accuracy,127.0.0.1,_Z15accuracy_kerneliiiPKfPKiPi,1.0,7.0,"(256, 1, 1)","(2048, 1, 1)",0.0,8.6,-1.0,-1.0,-1.0,"2,048",-1.0,-1.0,"44,344.29","42,483.88","686,880.26",319.07,-1.0,12.17,-1.0,6.07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,566433.0,accuracy,127.0.0.1,main_l57,1.0,13.0,"(128, 1, 1)","(2048, 1, 1)",0.0,8.6,-1.0,-1.0,-1.0,"2,048",-1.0,-1.0,"56,740.00","46,188.44","861,213.01",640.42,-1.0,17.64,-1.0,5.34,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,566547.0,accuracy,127.0.0.1,main_l57,1.0,13.0,"(128, 1, 1)","(2048, 1, 1)",0.0,8.6,-1.0,-1.0,-1.0,"2,048",-1.0,-1.0,"56,740.71","46,188.41","860,023.71",641.49,-1.0,20.88,-1.0,6.32,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,566656.0,accuracy,127.0.0.1,main_l57,1.0,13.0,"(128, 1, 1)","(2048, 1, 1)",0.0,8.6,-1.0,-1.0,-1.0,"2,048",-1.0,-1.0,"56,740.10","46,188.40","859,528.90",641.50,-1.0,18.77,-1.0,5.68,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,566735.0,ace,127.0.0.1,_Z14calculateForcePA400_A400_dS1_S1_S1_dddddd,1.0,7.0,"(8, 8, 4)","(50, 50, 100)",0.0,8.6,-1.0,-1.0,-1.0,"250,000",-1.0,-1.0,"1,365,062.16","669,550.59","50,960,093.99",910.45,-1.0,3.62,-1.0,1.49,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1.0,566735.0,ace,127.0.0.1,_Z9allenCahnPA400_A400_dS1_S1_S1_S1_S1_dddddddd,1.0,7.0,"(8, 8, 4)","(50, 50, 100)",0.0,8.6,-1.0,-1.0,-1.0,"250,000",-1.0,-1.0,"2,016,748.40","1,403,644.34","81,489,728.04",817.06,-1.0,4.64,-1.0,1.86,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,2.0,566735.0,ace,127.0.0.1,_Z21boundaryConditionsPhiPA400_A400_d,1.0,7.0,"(8, 8, 4)","(50, 50, 100)",0.0,8.6,-1.0,-1.0,-1.0,"250,000",-1.0,-1.0,"94,067.76","88,235.29","378,540.38",700.25,-1.0,32.81,-1.0,21.66,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,3.0,566735.0,ace,127.0.0.1,_Z15thermalEquationPA400_A400_dS1_S1_S1_dd

In [31]:
for col in df_merged.columns:
    print(col)

ID
Process ID
Process Name
Host Name
Kernel Name
Context
Stream
Block Size
Grid Size
Device
CC
FBSP.TriageSCG.dramc__read_throughput.avg.pct_of_peak_sustained_elapsed
FBSP.TriageSCG.dramc__throughput.avg.pct_of_peak_sustained_elapsed
FBSP.TriageSCG.dramc__write_throughput.avg.pct_of_peak_sustained_elapsed
FE_B.TriageAC.gr__ctas_launched_queue_sync.sum
LTS.TriageSCG.lts__average_t_sector_hit_rate_realtime.pct
LTS.TriageSCG.lts__throughput.avg.pct_of_peak_sustained_elapsed
SM_A.TriageAC.l1tex__data_pipe_lsu_wavefronts.avg
SM_A.TriageAC.l1tex__lsu_writeback_active.avg
SM_A.TriageAC.sm__cycles_active.avg
SM_A.TriageAC.sm__inst_executed_realtime.avg.per_cycle_active
SM_A.TriageSCG.l1tex__throughput.avg.pct_of_peak_sustained_elapsed
SM_A.TriageSCG.sm__inst_executed_pipe_alu_realtime.avg.pct_of_peak_sustained_elapsed
SM_B.TriageAC.l1tex__t_sector_hit_rate.pct
SM_C.TriageSCG.smsp__inst_executed_pipe_fmaheavy.avg.pct_of_peak_sustained_elapsed
SM_C.TriageSCG.smsp__inst_executed_pipe_fmalite.avg.